# 腾讯行情接口实验

本 notebook 使用腾讯公开行情页面背后的接口，覆盖：

1. 指定市场的标的范围；
2. 批量实时行情快照；
3. 当日分钟 K 线、历史日/周/月 K 线。

接口是网页接口而不是正式数据服务，示例默认低频、单连接请求。

In [ ]:
import json
import re
import time
from datetime import date, datetime
from collections.abc import Iterable

import httpx
import pandas as pd


client = httpx.Client(
    timeout=httpx.Timeout(connect=5.0, read=15.0, write=15.0, pool=15.0),
    limits=httpx.Limits(max_keepalive_connections=1, max_connections=1),
    trust_env=False,
    follow_redirects=True,
    headers={
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36"
        ),
    },
)


## 1. 市场标的范围

腾讯当前网页使用 `getBoardRankList` 返回沪深京 A 股榜单。接口按价格排序分页，代码端需要去重并按市场前缀过滤；`board_code=aStock` 返回的总数可用于停止分页。

In [ ]:
def normalize_symbol(value: str) -> str:
    raw_value = str(value).strip().lower()
    suffix_map = {"xshg": "sh", "xshe": "sz", "xbj": "bj"}
    if "." in raw_value:
        parts = raw_value.split(".")
        if len(parts) == 2 and parts[0].isdigit():
            return f"{suffix_map.get(parts[1], parts[1])}{parts[0]}"
    if raw_value.startswith(("sh", "sz", "bj")):
        return raw_value
    if len(raw_value) != 6 or not raw_value.isdigit():
        raise ValueError(f"无法识别标的代码: {value!r}")
    if raw_value.startswith(("600", "601", "603", "605", "688", "689", "900")):
        return f"sh{raw_value}"
    if raw_value.startswith(("000", "001", "002", "003", "200", "300", "301", "399")):
        return f"sz{raw_value}"
    if raw_value.startswith(("430", "440", "830", "831", "832", "833", "834", "835", "836", "837", "838", "839", "870", "871", "872", "873", "920")):
        return f"bj{raw_value}"
    raise ValueError(f"无法推断市场: {value!r}")


def symbol_in_market(symbol: str, market: str) -> bool:
    normalized_market = market.strip().lower()
    if normalized_market in {"cn_a", "a", "hsj"}:
        return symbol.startswith(("sh", "sz", "bj"))
    market_prefix = {"sh_a": "sh", "沪a": "sh", "sz_a": "sz", "深a": "sz", "bj_a": "bj", "北a": "bj"}[normalized_market]
    return symbol.startswith(market_prefix)


def tencent_universe(
    market: str = "cn_a",
    *,
    limit: int | None = None,
    page_size: int = 200,
    pause: float = 0.15,
) -> pd.DataFrame:
    rows: list[dict] = []
    seen_symbols: set[str] = set()
    offset = 0
    total = None
    while total is None or offset < total:
        response = client.get(
            "https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList",
            params={
                "_appver": "11.17.0",
                "board_code": "aStock",
                "sort_type": "price",
                "direct": "down",
                "offset": offset,
                "count": page_size,
            },
        )
        response.raise_for_status()
        payload = response.json()
        if payload.get("code") != 0:
            raise RuntimeError(payload)
        data = payload["data"]
        total = int(data.get("total", 0))
        rank_list = data.get("rank_list", [])
        for row in rank_list:
            symbol = normalize_symbol(row["code"])
            if not symbol_in_market(symbol, market) or symbol in seen_symbols:
                continue
            seen_symbols.add(symbol)
            rows.append(
                {
                    "symbol": symbol,
                    "code": symbol[2:],
                    "name": row.get("name", ""),
                    "stock_type": row.get("stock_type", ""),
                    "last": pd.to_numeric(row.get("zxj"), errors="coerce"),
                    "change_percent": pd.to_numeric(row.get("zdf"), errors="coerce"),
                }
            )
        if limit is not None and len(rows) >= limit:
            break
        if len(rank_list) < page_size:
            break
        offset += page_size
        if pause:
            time.sleep(pause)
    result = pd.DataFrame(rows).sort_values("symbol").reset_index(drop=True)
    return result.head(limit) if limit is not None else result


tencent_pool = tencent_universe("cn_a", limit=20)
tencent_pool.head()

## 2. 实时行情快照

快照接口是 `qt.gtimg.cn/q`，返回 GBK 编码的 `v_<symbol>="字段~字段..."` 文本。普通沪深股票的成交量通常按手返回，下面统一换算成股；科创板、北交所和指数保留其常见的股单位。成交额优先从 `价格/成交量/成交额` 复合字段读取。

In [ ]:
TENCENT_SNAPSHOT_PATTERN = re.compile(r'v_([a-z]{2}\d{6})="([^"]*)";', re.IGNORECASE)


def tencent_volume_to_shares(symbol: str, raw_volume):
    value = pd.to_numeric(raw_volume, errors="coerce")
    if pd.isna(value):
        return value
    if symbol.startswith(("bj", "sh688", "sh689", "sh000", "sz399")):
        return value
    return value * 100


def tencent_snapshot(symbols: str | Iterable[str], batch_size: int = 100) -> pd.DataFrame:
    requested = [symbols] if isinstance(symbols, str) else list(symbols)
    normalized_symbols = list(dict.fromkeys(normalize_symbol(symbol) for symbol in requested))
    parsed: dict[str, dict] = {}
    for start_index in range(0, len(normalized_symbols), batch_size):
        batch = normalized_symbols[start_index : start_index + batch_size]
        response = client.get("https://qt.gtimg.cn/q", params={"q": ",".join(batch)})
        response.raise_for_status()
        text = response.content.decode("gbk", errors="replace")
        for match in TENCENT_SNAPSHOT_PATTERN.finditer(text):
            symbol = normalize_symbol(match.group(1))
            fields = match.group(2).split("~")
            if len(fields) < 36:
                continue
            composite = fields[35].split("/")
            amount = pd.to_numeric(composite[2], errors="coerce") if len(composite) >= 3 else pd.NA
            parsed[symbol] = {
                "symbol": symbol,
                "code": fields[2],
                "name": fields[1],
                "timestamp": pd.to_datetime(fields[30], format="%Y%m%d%H%M%S", errors="coerce"),
                "last": pd.to_numeric(fields[3], errors="coerce"),
                "previous_close": pd.to_numeric(fields[4], errors="coerce"),
                "open": pd.to_numeric(fields[5], errors="coerce"),
                "high": pd.to_numeric(fields[33], errors="coerce"),
                "low": pd.to_numeric(fields[34], errors="coerce"),
                "change": pd.to_numeric(fields[31], errors="coerce"),
                "change_percent": pd.to_numeric(fields[32], errors="coerce"),
                "bid": pd.to_numeric(fields[9], errors="coerce"),
                "ask": pd.to_numeric(fields[19], errors="coerce"),
                "volume": tencent_volume_to_shares(symbol, fields[6]),
                "amount": amount,
                "raw_fields": fields,
            }
    return pd.DataFrame(
        [parsed[symbol] for symbol in normalized_symbols if symbol in parsed]
    )


tencent_quotes = tencent_snapshot(["sh600519", "sz000001"])
tencent_quotes

## 3. 当日分钟 K 和历史 K

分钟接口使用 `appstock/app/kline/mkline`；日/周/月接口使用 `newfqkline/get`。腾讯历史接口单次最多返回约 640 根，跨年份时按年度请求并去重。日线的成交量按手返回时换算成股，成交额字段按万元返回时换算成元。

In [ ]:
TENCENT_MINUTES = {"1m": 1, "5m": 5, "15m": 15, "30m": 30, "60m": 60}
TENCENT_UNITS = {"1d": "day", "1w": "week", "1M": "month"}


def embedded_json(text: str):
    decoder = json.JSONDecoder()
    positions = sorted(position for marker in ("[", "{") if (position := text.find(marker)) >= 0)
    for position in positions:
        try:
            return decoder.raw_decode(text[position:])[0]
        except json.JSONDecodeError:
            continue
    raise ValueError("响应中没有找到 JSON")


def tencent_minute_kline(symbol: str, interval: str = "5m", limit: int = 240) -> pd.DataFrame:
    normalized_symbol = normalize_symbol(symbol)
    minutes = TENCENT_MINUTES[interval]
    response = client.get(
        "https://ifzq.gtimg.cn/appstock/app/kline/mkline",
        params={"param": f"{normalized_symbol},m{minutes},,{limit}"},
    )
    response.raise_for_status()
    payload = embedded_json(response.text)
    rows = payload.get("data", {}).get(normalized_symbol, {}).get(f"m{minutes}", [])
    records = []
    for row in rows:
        if len(row) < 6:
            continue
        records.append(
            {
                "datetime": pd.to_datetime(row[0], format="%Y%m%d%H%M", errors="coerce"),
                "open": pd.to_numeric(row[1], errors="coerce"),
                "close": pd.to_numeric(row[2], errors="coerce"),
                "high": pd.to_numeric(row[3], errors="coerce"),
                "low": pd.to_numeric(row[4], errors="coerce"),
                "volume": tencent_volume_to_shares(normalized_symbol, row[5]),
                "turnover_percent": pd.to_numeric(row[7], errors="coerce") if len(row) > 7 else pd.NA,
            }
        )
    return pd.DataFrame(records).sort_values("datetime").reset_index(drop=True)


def tencent_history_kline(
    symbol: str,
    interval: str = "1d",
    *,
    start_date: str | date | datetime | None = None,
    end_date: str | date | datetime | None = None,
    adjust: str = "",
    limit: int = 500,
) -> pd.DataFrame:
    normalized_symbol = normalize_symbol(symbol)
    unit = TENCENT_UNITS[interval]
    end_value = pd.Timestamp(end_date or date.today()).date()
    start_value = pd.Timestamp(start_date or f"{end_value.year}-01-01").date()
    rows: list[list] = []
    for request_year in range(start_value.year, end_value.year + 1):
        response = client.get(
            "https://proxy.finance.qq.com/ifzqgtimg/appstock/app/newfqkline/get",
            params={
                "_var": f"kline_{unit}{adjust}{request_year}",
                "param": f"{normalized_symbol},{unit},{request_year}-01-01,{request_year + 1}-12-31,640,{adjust}",
                "r": "0.8205512681390605",
            },
        )
        response.raise_for_status()
        payload = embedded_json(response.text)
        symbol_data = payload.get("data", {}).get(normalized_symbol, {})
        candidate_keys = [f"{adjust}{unit}" if adjust else unit, unit, "day"]
        selected_rows = next((symbol_data[key] for key in candidate_keys if isinstance(symbol_data.get(key), list)), [])
        rows.extend(selected_rows)
    records = []
    for row in rows:
        if len(row) < 6:
            continue
        records.append(
            {
                "datetime": pd.to_datetime(row[0], errors="coerce"),
                "open": pd.to_numeric(row[1], errors="coerce"),
                "close": pd.to_numeric(row[2], errors="coerce"),
                "high": pd.to_numeric(row[3], errors="coerce"),
                "low": pd.to_numeric(row[4], errors="coerce"),
                "volume": tencent_volume_to_shares(normalized_symbol, row[5]),
                "turnover_percent": pd.to_numeric(row[7], errors="coerce") if len(row) > 7 else pd.NA,
                "amount": pd.to_numeric(row[8], errors="coerce") * 10000 if len(row) > 8 else pd.NA,
            }
        )
    result = pd.DataFrame(records)
    if result.empty:
        return result
    result = result[(result["datetime"].dt.date >= start_value) & (result["datetime"].dt.date <= end_value)]
    return result.sort_values("datetime").drop_duplicates("datetime").tail(limit).reset_index(drop=True)


tencent_today_5m = tencent_minute_kline("sh600519", "5m", limit=48)
tencent_daily = tencent_history_kline("sh600519", "1d", limit=10)
tencent_today_5m.tail(), tencent_daily.tail()

### 观察结论

- 标的池：腾讯榜单接口页大小较大，适合顺序分页；结果仍要去重和按代码排序。
- 快照：腾讯文本接口支持批量代码，字段比新浪更长，但前 35 个字段足以组成常用快照。
- K 线：分钟接口适合当日数据；历史日线单次最多约 640 根，跨年份请求后去重。
- 复权：日/周/月接口可传空字符串、`qfq` 或 `hfq`；分钟接口按原始行情处理。